In [1]:
from modelscope import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

C:\Users\17246\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-08-15 15:21:50,826 - modelscope - INFO - Creating symbolic link [C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0.6B].
2025-08-15 15:21:50,828 - modelscope - WARNING - Failed to create symbolic link C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0.6B for C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0___6B.


2025-08-15 15:21:58,526 - modelscope - INFO - Creating symbolic link [C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0.6B].
2025-08-15 15:21:58,530 - modelscope - WARNING - Failed to create symbolic link C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0.6B for C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0___6B.


In [2]:
data = [
    {"question": "你是谁", "think": "我是谁，我要思考一下，我属于一个学习助手", "answer": "我是你的学习助手"},
    {"question": "你是谁开发的", "think": "我要思考一下，我由慧科团队开发", "answer": "我是由慧科团队开发的"},
    {"question": "你是一个学习助手", "think": "我要思考一下，我是一个学习助手", "answer": "我是一个学习助手"}
]

In [3]:
from torch.utils.data import Dataset, DataLoader

lora_prompt_templates = """
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>

</think>
{answer}<|im_end|>
"""


class dataset(Dataset):
    def __init__(self, data, max_length=128):
        self.encodings = []
        for qa in data:
            text = lora_prompt_templates.format(question=qa['question'], answer=qa['answer'])
            encoded = tokenizer(text, max_length=max_length, truncation=True, padding='max_length',
                                return_tensors='pt')
            inpus_ids = encoded['input_ids'].squeeze()
            self.encodings.append(inpus_ids)

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]


dataset = dataset(data)
dataloader = DataLoader(dataset, batch_size=2)


In [7]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj"],
)

lora_model = get_peft_model(model, lora_config)


In [ ]:
from torch import optim

optimizer = optim.Adam(lora_model.parameters(), lr=1e-4)

for epoch in range(100):
    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch
        outputs = lora_model(input_ids, labels=input_ids)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        print(loss.item())


8.38269329071045
7.513036251068115
6.653998851776123
5.698681831359863
5.032700061798096
3.9197866916656494
3.558908224105835
2.6365649700164795
2.4871602058410645
2.093597888946533
1.9808712005615234
1.894180417060852
1.7818506956100464
1.7877600193023682
1.6819263696670532
1.7140485048294067
1.6171332597732544
1.6644868850708008
1.5658787488937378
1.6220229864120483
1.5185632705688477
1.569916009902954
1.4644103050231934
1.494810938835144
1.387022614479065
1.385738492012024
1.3125489950180054
1.3257752656936646
1.2470906972885132
1.274620532989502
1.1982929706573486
1.20842444896698
1.1443058252334595
1.1016838550567627
1.0720539093017578
1.0194779634475708
1.0186759233474731
0.9704051613807678
0.9628602266311646
0.9196091294288635
0.9081122279167175
0.8664413094520569


In [ ]:
prompt = "你是谁"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt")
generated_ids = lora_model.generate(**model_inputs, max_new_tokens=32768)
content = tokenizer.decode(generated_ids[0])
print(content)

In [ ]:
lora_model.save_pretrained("../model/loraQwen3")

In [ ]:
from modelscope import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = PeftModel.from_pretrained(model, "../model/loraQwen3")

In [ ]:
prompt = "你能做什么"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt")
generated_ids = model.generate(**model_inputs, max_new_tokens=32768)
content = tokenizer.decode(generated_ids[0])
print(content)

In [ ]:
prompt = "你是谁"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
print(text)


In [ ]:
model_inputs = tokenizer([text], return_tensors="pt")
generated_ids = model.generate(**model_inputs, max_new_tokens=32768)
content = tokenizer.decode(generated_ids[0])
print(content)


In [2]:
import json

data = []
target = 200
count = 0
with open('../data/Finance_R1-Distill_Data/Finance_R1-Distill_data_0.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        data.append(item)
        count += 1
        if count >= target:
            break

data[:100]

[{'id': 1,
  'user_input': "Explain tradeoffs between fiscal and monetary policy as tools in a nation's economic toolkit. Provide examples of past instances when each were utilized, the economic conditions that led to them being deployed, their intended effects, and an evaluation of their relative efficacy and consequences.",
  'reasoning_content': "\nOkay, so I need to explain the tradeoffs between fiscal and monetary policy as tools in a nation's economic toolkit. Hmm. Let me start by recalling what fiscal and monetary policies are. Fiscal policy is about government spending and taxation, right? So when the government adjusts its spending levels or tax rates, that's fiscal policy. Monetary policy, on the other hand, is managed by the central bank and involves controlling the money supply and interest rates. The Federal Reserve in the US, for example, uses tools like changing interest rates or buying/selling government bonds.\n\nThe question is about the tradeoffs between these two. S

In [3]:


from torch.utils.data import Dataset, DataLoader

lora_prompt_templates = """
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>
{think}
</think>
{answer}<|im_end|>
"""


class dataset(Dataset):
    def __init__(self, data, max_length=2048):
        self.encodings = []
        for qa in data:
            text = lora_prompt_templates.format(question=item["user_input"], think=item["reasoning_content"], answer=item["answer_r1"])
            encoded = tokenizer(text, max_length=max_length, truncation=True, padding='max_length',
                                return_tensors='pt')
            input_ids = encoded['input_ids'].squeeze()
            self.encodings.append(input_ids)

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]


train_dataset = dataset(data)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
for batch in train_loader:
    print(batch)
    break


tensor([[   198, 151644,    872,  ..., 151643, 151643, 151643],
        [   198, 151644,    872,  ..., 151643, 151643, 151643],
        [   198, 151644,    872,  ..., 151643, 151643, 151643],
        ...,
        [   198, 151644,    872,  ..., 151643, 151643, 151643],
        [   198, 151644,    872,  ..., 151643, 151643, 151643],
        [   198, 151644,    872,  ..., 151643, 151643, 151643]])


In [ ]:
print(model)

In [4]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=2,
    target_modules=["q_proj", "v_proj", "k_proj"]
)
lora_model = get_peft_model(model, lora_config)

lora_model.print_trainable_parameters()


trainable params: 401,408 || all params: 596,451,328 || trainable%: 0.0673


In [ ]:
print(model)

In [5]:
import torch

optimizer = torch.optim.Adam(lora_model.parameters(), lr=0.0001)

step = 100
for epoch in range(step):
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = lora_model(batch, labels=batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(loss.item())

KeyboardInterrupt: 

In [ ]:
prompt = "Explain tradeoffs between fiscal and monetary policy as tools in a nation's economic toolkit. Provide examples of past instances when each were utilized, the economic conditions that led to them being deployed, their intended effects, and an evaluation of their relative efficacy and consequences."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)
model_inputs = tokenizer([text], return_tensors="pt")
generated_ids = lora_model.generate(**model_inputs, max_new_tokens=32768)
content = tokenizer.decode(generated_ids[0])
print(content)


In [ ]:
lora_model.save_pretrained("../models/loraQwen3")


In [ ]:

from modelscope import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = PeftModel.from_pretrained(model, "../models/loraQwen3")


In [ ]:
prompt = "你能做什么"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt")
generated_ids = model.generate(**model_inputs, max_new_tokens=32768)
content = tokenizer.decode(generated_ids[0])
print(content)